# Lab 00 — Observability Stack

## 終點：完整自動化 AIOps Pipeline

每個 lab 不是孤立的練習 — 它是這個 pipeline 裡的一個元件。
先看清楚整張圖，你才知道自己在哪裡、做的事情會去哪裡。

```
網路設備（交換器 / 路由器 / 防火牆）
      │  SNMP polling → RRD 儲存 → 週期性轉成 CSV
      ▼
synthetic_rrd_metrics.csv  ← 課程使用此合成資料模擬真實環境
      │
      ▼ infra/exporter.py（回放 CSV 成即時 Prometheus metrics）
      │
      ▼  每 5 秒 scrape /metrics
┌──────────────────────────────────────────────────────────────┐
│  Prometheus                                                  │
│                                                              │
│  Recording rules  ←──────────── Lab 01 特徵工程               │
│    network:traffic_mean_1h         rolling mean / std        │
│    network:error_rate              safe_div(errors, ucast)   │
│    network:traffic_zscore          z-score 公式              │
│                                                              │
│  Alert rules ─────────────────── Lab 02 z-score 偵測          │
│    流量Surge: z > 3             驗證閾值                  │
│    ErrorRateSurge: z > 3                                     │
│    BroadcastStorm: z > 3 for 30s                             │
│                                                              │
│  Alert rules ─────────────────── Lab 03 SPC 管制圖            │
│    metric > UCL                    驗證管制線                │
│    metric < LCL                                              │
│                                                              │
│  ML scoring service ───────────── Lab 04 ML 異常偵測           │
│    每 5 秒 pull Prometheus API     訓練 Isolation Forest      │
│    → model.predict(X)             → 匯出 model.pkl           │
│    → push anomaly_score{}                                    │
│    Alert rules: anomaly_score > 0.7                          │
│                                                              │
│  forecast.py daemon ───────────── Lab 06 預測                 │
│    → push predicted_traffic{}      訓練 SARIMA / LSTM        │
│    Alert rules: 偏離預測 > N σ                               │
└──────────────────────────┬───────────────────────────────────┘
                           │
          ┌────────────────▼────────────────┐
          │  Alertmanager  ←──────────────── Lab 05 告警降噪      │
          │  inhibit_rules                   關聯分析、抑制規則  │
          │  route.group_by                  grouping 邏輯       │
          └────────────────┬────────────────┘
                           │ 去重、關聯後的告警
                           ▼
                  ┌────────────────┐
                  │   Grafana      │
                  │  ──────────    │
                  │  即時 metrics   │
                  │  ⚡ Anomaly    │
                  │    annotations │
                  │  🔴 Alert list │
                  └───────┬────────┘
                          │
                          ▼
          ⚠ 你（人類）：確認告警、判讀業務背景
                  │
                  │ 疑難狀況時
                  ▼
        LLM runbook assistant  ←──── Lab 07 RCA + LLM
          alert context → LLM API     context 設計
          → diagnosis + action steps  → 診斷準確率評估
```

---

## 四條整合路徑

| 路徑 | 適用 | Labs | 實作方式 |
|---|---|---|---|
| **1 統計直譯** | 公式可用 PromQL 表達 | Lab 01, Lab 02, Lab 03 | 寫進 `infra/prometheus/alerts.yml`，Prometheus 原生計算，Lab 08 部署 |
| **2 ML sidecar** | 需要 ML 模型推論 | Lab 04, Lab 06 | future scoring / forecasting service：pull → 推論 → push 結果回 Prometheus |
| **3 Alertmanager** | 告警路由 / 抑制 / 關聯 | Lab 05 | `alertmanager.yml`：`inhibit_rules`、`route.group_by` |
| **4 LLM webhook** | 自然語言 RCA | Lab 07 | Alert webhook → Python → LLM API → Slack / PagerDuty |

---

## 學習策略：以終為始

你在**批次分析軌**（Labs 01–07）開發與驗證每個元件；
通過 ⚠ 人類驗證點後，部署到**即時監控軌**自動執行。

```
Lab               批次驗證的事                整合路徑    即時軌對應
──────────────────────────────────────────────────────────────────────
Lab 01 特徵工程    rolling mean/std, 比例特徵    路徑 1     Recording rules
Lab 02 Baseline    z-score 閾值、recall          路徑 1     Alert rules（Lab 08）
Lab 03 SPC         UCL / LCL 管制線             路徑 1     Alert rules（UCL hard-coded）
Lab 04 ML 偵測     Isolation Forest 訓練         路徑 2     scoring service sidecar
Lab 05 告警降噪    關聯分析、inhibition 規則      路徑 3     Alertmanager config
Lab 06 預測        SARIMA / LSTM forecast        路徑 2     forecast.py sidecar
Lab 07 RCA + LLM   context 設計、診斷準確率      路徑 4     LLM webhook
Lab 08 部署        Lab 01-02 邏輯完整部署驗證      —          end-to-end 驗證
```

**Lab 08 示範路徑 1 的完整部署流程**，是最小可驗證的 end-to-end 案例。
路徑 2–4 架構模式相同，規模更大，延伸到 ML 服務化時再展開。


In [ ]:
from pathlib import Path
from IPython.display import SVG, display

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for base in (start, *start.parents):
        if (base / "environment.yml").exists():
            return base
    raise FileNotFoundError("Could not find project root containing environment.yml")

svg_candidates = [
    Path("diagrams/aiops_pipeline_overview.svg"),
    find_project_root() / "labs/self-study" / "diagrams/aiops_pipeline_overview.svg",
]
for svg_path in svg_candidates:
    if svg_path.exists():
        display(SVG(filename=str(svg_path)))
        break
else:
    raise FileNotFoundError("Could not find diagram: diagrams/aiops_pipeline_overview.svg")


## Lab 00 概覽

### 學習目標

1. 理解 Prometheus pull model：scrape 是什麼、間隔為何重要
2. 使用 PromQL 查詢時間序列資料並轉成 DataFrame
3. 驗證 exporter → Prometheus → Grafana 三層資料流是否正常
4. 瞭解批次分析軌（notebook）與即時監控軌（Prometheus rules）的關係

### 前置條件

無。這是第一個 lab。需要在本機或課程環境已啟動 exporter、Prometheus、Grafana。

## 設計主線：先證明資料流可信

本章的實務連結是監控系統的第一個失敗點：資料沒有進來、label 不一致、scrape interval 不合適，後面的演算法都會建立在錯誤輸入上。

本章請用架構角度思考三件事：

1. **資料來源**：exporter、Prometheus、Grafana 各自負責什麼？哪一層壞掉時，你要怎麼定位？
2. **取樣頻率**：scrape interval 越短越敏感，但儲存與查詢成本也越高。你的異常通常持續幾秒、幾分鐘，還是幾小時？
3. **label schema**：device、port、role 這些 label 會決定後續能否做 grouping、RCA 與 dashboard drill-down。

設計原則：先讓資料可觀測，再讓模型可用。沒有可信 telemetry，就沒有可信 AIOps。


## 這門課的立場：人類驗證，AI 加速

AIOps 工具鏈——包含本課程的每一個 lab——是**決策支援工具，不是決策工具**。

| AI 擅長 | 人類不可或缺 |
|---|---|
| 同時監控數千個 metrics，不間斷 | 判斷這個 flag 是否代表真正的問題 |
| 毫秒內計算 z-score、SPC 管制線、ML 異常分數 | 提供業務背景：計畫維護、預期流量、已知事件 |
| 跨 port 比較，找出同步異常型態 | 設定符合業務容忍度的閾值 |
| 自動比對歷史 baseline | 決定要不要啟動應急流程 |

**每個 lab 都有「⚠ 人類驗證點」標記。遇到時請停下來思考，不要只是執行下一個 cell。**

AI 在這個 pipeline 裡是你的一線偵測員：速度快、不會疲勞、不會遺漏。
你是資深工程師：有 context、有判斷力、有責任。兩者缺一不可。

---

## 課程 Pipeline 全覽

```text
網路設備（SNMP / RRD）
    ↓
CSV 資料（synthetic_rrd_metrics.csv）
    ↓ 兩條平行軌道
    ├─ 即時監控軌（Lab 00）
    │      infra/exporter.py → Prometheus → Grafana
    │      運維團隊看到的即時儀表板
    │
    └─ 批次分析軌（Labs 01–07）
           特徵工程 → 異常偵測 → 告警降噪 → 預測 → RCA
           你在這裡開發和驗證偵測邏輯

                    ↑ 本 lab 把兩條軌道接起來
```

本 lab 聚焦在即時監控軌：讓 exporter、Prometheus、Grafana 都跑起來，
並示範兩個軌道如何對應到同一份資料。


## 課程路線圖

| Lab | 主題 | 你會做什麼 | AI 做什麼 | 人類驗證什麼 |
|---|---|---|---|---|
| **00** | 可觀測性堆疊 | 啟動 exporter、驗證監控 | 無 | 確認資料流通 |
| **01** | 時間序列特徵工程 | 計算 rolling stats、比例特徵 | 無 — 這步驟是你設計特徵 | 特徵是否有意義 |
| **02** | Baseline 異常偵測 | 設定 z-score 閾值 | 標記超出 baseline 的點 | 調整閾值、評估誤報 |
| **03** | SPC 管制圖 | 選擇管制規則 | 計算管制線、違規點 | 管制規則是否適合此業務 |
| **04** | ML 異常偵測 | 調整 contamination 參數 | Isolation Forest 評分 | 評估分數分布、決定切點 |
| **05** | 告警降噪 | 設定降噪規則 | 合併、抑制重複告警 | 確認不遺漏真實事件 |
| **06** | 預測 | 選擇預測視窗 | 時間序列預測 | 預測誤差是否在可接受範圍 |
| **07** | RCA + LLM | 撰寫 LLM prompt | 生成根因候選清單 | 評估 LLM 建議是否合理 |

**下一步 → `01_time_series_features.ipynb`**
| **08** | 部署到生產環境 | 載入 alert rules、設定 Grafana annotations | 24/7 執行 z-score 偵測 | 驗證 alert 確實觸發，決定是否部署 ML sidecar |

---
## 以下：可觀測性堆疊實作

把 exporter、Prometheus、Grafana 跑起來，讓課程的兩條軌道都有資料。


---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 1 — 啟動資料 exporter

`infra/exporter.py` 讀取 `data/synthetic/synthetic_rrd_metrics.csv`，
以加速模式回放成 Prometheus 可抓取的即時 metrics。

**在新的終端機視窗執行（不要在 Jupyter 裡執行，它會佔住 kernel）：**

```bash
cd aiops-anomaly-zero-to-hero
conda activate aiops-anomaly-zero-to-hero
python infra/exporter.py
```

預設 `REPLAY_SPEED_X=3600`：1 真實秒 = 1 模擬小時，30 天資料約 20 分鐘跑完。

啟動後，下面的 cell 可以用來確認 exporter 已在線上。


In [ ]:
import urllib.request, json, textwrap, time

def check_url(url: str, label: str) -> bool:
    try:
        with urllib.request.urlopen(url, timeout=3) as r:
            print(f"✅ {label} — {url}  (HTTP {r.status})")
            return True
    except Exception as e:
        print(f"❌ {label} — {url}\n   {e}")
        return False

print("檢查服務狀態…")
check_url("http://localhost:8000/metrics",  "Exporter   ")
check_url("http://localhost:9090/-/healthy","Prometheus ")
check_url("http://localhost:3000/api/health","Grafana Local")
# Grafana Cloud（選用）— 在瀏覽器開啟 https://<YOUR_ORG>.grafana.net 確認


## Step 2 — 預覽 exporter 輸出的 metrics

每個 Prometheus metric 由名稱 + label 組成：

```
network_rrd_inoctets{device_id="edge-fw-01", port_id="port-id7427", port_role="firewall"} 1234567.0
```

- **名稱** `network_rrd_inoctets` → 對應 CSV 欄位 `INOCTETS`
- **label** → 對應 CSV 的 `device_id`、`port_id`、`port_role` 欄位
- **值** → 該時間點的計數器數值

下面顯示前 40 行，讓你看清楚格式。


In [ ]:
try:
    with urllib.request.urlopen("http://localhost:8000/metrics", timeout=3) as r:
        lines = r.read().decode().splitlines()

    # 只顯示非 # 行（實際 metric 值）
    data_lines = [l for l in lines if not l.startswith("#") and l.strip()]
    for line in data_lines[:40]:
        print(line)
    print(f"\n... total {len(data_lines)} metric samples")
except Exception as exc:
    print("Exporter is not reachable at http://localhost:8000/metrics.")
    print("Start it from the repository root with: python infra/exporter.py")
    print(f"Details: {type(exc).__name__}: {exc}")


---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明約 5–10 分鐘，請先不要執行 cell。

---

## 📖 概念：Prometheus Pull Model and PromQL

### Prometheus 的抓取（scrape）原理

大多數監控系統是 **push model**：應用程式主動把 metrics 推送到收集器。
Prometheus 反過來：它**定期去抓**（pull / scrape）每個 target 的 `/metrics` endpoint。

```
Prometheus                target（infra/exporter.py）
    │                           │
    │  GET /metrics (每 5 秒)   │
    ├──────────────────────────►│
    │                           │
    │◄──────────────────────────┤
    │  text/plain metrics       │
    │                           │
    ▼
TSDB（本地時間序列資料庫）
```

**Pull model 的優點**

- Prometheus 知道哪些 target 不回應（push model 中，沉默的 target 無法區分是正常還是掛掉）
- 集中控制 scrape 間隔，不需要每個應用自己實作推送排程
- 容易橫向擴展：只需新增 target，Prometheus 自動開始抓

**Scrape 間隔的影響**

scrape 間隔決定你能「看到」多細的變化。本課程設定 `scrape_interval: 5s`，
表示每 5 秒取一個樣本。如果一個事件持續 30 秒但你的間隔是 60 秒，事件可能被完全錯過。

---

### PromQL：時間序列的 SQL

PromQL（Prometheus Query Language）是專為時間序列設計的查詢語言。
最直接的類比：**PromQL 之於 Prometheus，就如同 SQL 之於關聯式資料庫**——但查詢的是「值如何隨Time變化」而非「哪幾列符合條件」。

**三個基本元素**

| 元素 | 說明 | 範例 |
|---|---|---|
| Metric name | 要查哪個指標 | `network_rrd_inoctets` |
| Label filter | 篩選特定 target | `{port_id="port-id7427"}` |
| Time函數 | 在時間軸上計算 | `rate(...[5m])`, `avg_over_time(...[1h])` |

**常用函數**

```promql
# 過去 5 分鐘的平均速率（將 counter 轉成 rate）
rate(network_rrd_inoctets{port_id="port-id7427"}[5m])

# 過去 1 小時的平均值
avg_over_time(network_rrd_inoctets{port_id="port-id7427"}[1h])

# 跨所有 port 計算 IN + OUT 總流量
network_rrd_inoctets + network_rrd_outoctets
```

**直觀理解**：想像每個 metric + label 組合是一份試算表的欄位，Time是列。
PromQL 讓你在這個多維試算表上做滑動窗口計算，不需要把資料拉到 Python 再算。

**適用條件**

PromQL 最適合「數值如何隨Time變化」的問題。它不適合複雜的 join 或多步驟的統計邏輯（那些更適合在 Python/pandas 裡做，這也是為什麼課程同時有兩條軌道）。

## Step 3 — Prometheus 與 PromQL 入門

Prometheus 提供 HTTP API，可以用 PromQL 查詢：

```
http://localhost:9090/api/v1/query?query=<PromQL 表達式>
```

下面的 cell 示範兩個常用查詢，以及如何把結果轉成 pandas DataFrame。


---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

In [ ]:
import urllib.parse, pandas as pd

def promql(expr: str) -> pd.DataFrame:
    """Run a PromQL instant query and return a tidy DataFrame."""
    url = "http://localhost:9090/api/v1/query?query=" + urllib.parse.quote(expr)
    with urllib.request.urlopen(url, timeout=5) as r:
        result = json.loads(r.read())
    if result["status"] != "success":
        raise RuntimeError(result)
    rows = []
    for item in result["data"]["result"]:
        row = {**item["metric"]}
        row["value"] = float(item["value"][1])
        rows.append(row)
    return pd.DataFrame(rows)

# 查詢：所有 port 當前的 INOCTETS
try:
    df_current = promql("network_rrd_inoctets")
    print("PromQL: network_rrd_inoctets")
    expected_cols = ["device_id", "port_id", "port_role", "value"]
    if df_current.empty:
        print("No network_rrd_inoctets samples were returned.")
        print("Start infra/exporter.py and make sure Prometheus uses infra/prometheus/prometheus.yml.")
    elif all(col in df_current.columns for col in expected_cols):
        print(df_current[expected_cols].to_string(index=False))
    else:
        print("Prometheus returned data, but labels differ from the course exporter output:")
        print(df_current.head().to_string(index=False))
except Exception as exc:
    print("Prometheus query skipped.")
    print("Start Prometheus with infra/prometheus/prometheus.yml, then rerun this cell.")
    print(f"Details: {type(exc).__name__}: {exc}")


In [ ]:
# 同一份資料，用 pandas 從 CSV 讀取
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "environment.yml").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

csv_df = pd.read_csv(PROJECT_ROOT / "data/synthetic/synthetic_rrd_metrics.csv",
                     parse_dates=["timestamp"])

# 最新一筆（對應 exporter 目前回放到的Time點）
latest = csv_df.sort_values("timestamp").groupby("port_id").last().reset_index()
print("CSV 最新一筆 INOCTETS（對應 exporter 目前位置）：")
print(latest[["device_id","port_id","port_role","INOCTETS"]].to_string(index=False))
print()
print("Prometheus 和 CSV 的欄位對應：")
print("  PromQL metric name       → CSV column")
print("  network_rrd_inoctets     → INOCTETS")
print("  network_rrd_outerrors    → OUTERRORS")
print("  network_rrd_indiscards   → INDISCARDS")
print("  network_rrd_inbroadcastpkts → INBROADCASTPKTS")
print("  … 共 15 個 metric，對應 15 個 CSV 欄位")


---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 4 — 開啟 Grafana Local 儀表板

在瀏覽器開啟 `http://localhost:3000`，登入（預設帳號 admin / 密碼 admin）。

初次登入後：
1. 左側選單 → **Dashboards**
2. 找到已匯入的 **Network Interface Monitoring** 儀表板

儀表板的每個 panel 對應一個 PromQL 查詢，這些查詢和你在 Lab 01 裡用 pandas 計算的特徵是同一件事：

| Grafana panel | PromQL | Lab 01 計算 |
|---|---|---|
| 流量 (bytes/5min) | `network_rrd_inoctets + network_rrd_outoctets` | `df["traffic_total"]` |
| Error rate | `network_rrd_inerrors / network_rrd_inucastpkts` | `df["error_rate"]` |
| Discard rate | `network_rrd_indiscards / network_rrd_inucastpkts` | `df["discard_rate"]` |
| Broadcast packets | `network_rrd_inbroadcastpkts` | `df["broadcast_total"]` |

Grafana 讓你即時看到這些值；Lab 02–4 讓你在歷史資料上偵測哪些時間點超出正常範圍。

---

## ⚠ 人類驗證點 #0

在繼續之前，確認你能在 Grafana 儀表板上看到資料。

問自己：
1. 流量 圖上的曲線有規律的日夜變化嗎？（正常的商業網路應該有）
2. 目前有沒有明顯的 error_rate 或 discard_rate 升高？
3. 如果你是運維人員，你會對哪個 panel 最先設告警？

這些直覺判斷，是後面幾個 lab 裡 閾值 設計的起點。**AI 不知道你的業務容忍度；你才知道。**


In [ ]:
# Reload Prometheus config after changes (無需重啟)
try:
    with urllib.request.urlopen(
        urllib.request.Request("http://localhost:9090/-/reload", method="POST"), timeout=5
    ) as r:
        print(f"Prometheus config reloaded: HTTP {r.status}")
except Exception as e:
    print(f"Reload skipped or failed (OK if Prometheus just started): {e}")


---
💬 **討論暫停 ▸ 全班討論** — 停止執行，分享你的觀察。

---

## 兩條軌道如何整合：Lab 邏輯 → 自動化監控

批次分析軌的每個 lab 都在驗證一個偵測邏輯；驗證通過後，同樣的邏輯可以透過三條路徑部署到即時監控軌。

```
批次分析軌（Labs 01–07）                   即時監控軌（整合路徑）
───────────────────────────               ──────────────────────────────────────

Lab 01 特徵工程                             路徑 1 — 統計規則直譯
  rolling mean / std                 →   infra/prometheus/alerts.yml (recording rules)
  safe_div(errors, ucast)                  network:traffic_mean_1h
  ─ PromQL 可原生計算相同公式 ─              network:error_rate_zscore

Lab 02 Baseline z-score                     路徑 1（續）— Alert rules
  z > 3 → flag                      →   alert 流量Surge: expr: network:traffic_zscore > 3
  ─ 閾值值從 lab 驗證 ─               ⬆ Lab 08 把這步驟自動完成

Lab 03 SPC 管制圖                           路徑 1（續）
  UCL / LCL                         →   alert rules：metric > UCL（hard-coded from Lab 03）

Lab 04 ML 異常偵測                          路徑 2 — ML 評分服務（sidecar）
  Isolation Forest.fit(X)           →   scoring service：
  model.decision_function(x)              每 5 秒 pull Prometheus API
  ─ PromQL 不能跑 ML 模型 ─               → 跑 isolation_forest.pkl
                                          → push anomaly_score{port_id=...} 回 Prometheus
                                          → alert rule: anomaly_score > 0.7

Lab 05 告警降噪                             路徑 3 — Alertmanager config
  correlation, inhibition           →   alertmanager.yml: inhibit_rules + route.group_by
  ─ 告警路由/抑制交給 Alertmanager ─

Lab 06 預測                                 路徑 2（續）— Forecast 服務
  SARIMA / LSTM forecast            →   forecast.py daemon：推 predicted_{metric}
                                          → alert rule: 實際值偏離預測值 > N σ

Lab 07 RCA + LLM                            路徑 4 — LLM runbook assistant
  context → LLM → diagnosis         →   alert webhook → Python → LLM API → Slack/PagerDuty

⚠ 每個箭頭都需要「人類驗證點」通過後，才應該部署到生產環境
```

### 路徑 1 vs 路徑 2 的關鍵差別

| | 路徑 1（直譯） | 路徑 2（ML sidecar）|
|---|---|---|
| 適用 | 統計公式（mean, std, z-score, 比率） | ML 模型推論（Isolation Forest, LSTM）|
| 執行位置 | Prometheus 內部 | 獨立 Python 服務 |
| 部署方式 | 修改 `infra/prometheus/alerts.yml` | 另建 scoring service，定期讀取 Prometheus API 並回寫分數 |
| 資料流 | exporter → Prometheus scrape → PromQL eval | exporter → Prometheus → HTTP API → scorer → Pushgateway → Prometheus |
| Lab 08 涵蓋？ | 是（Lab 02 → alert rules） | 下一步（需要另建 scoring service）|

本課程的 Lab 08 完成路徑 1（z-score → alert rules）的完整部署示範。
路徑 2–4 是真實生產環境的下一步，架構模式相同，規模更大。


In [ ]:
# 驗證：目前 Prometheus 已部署哪些 recording / alert rules？
import urllib.request, json

def api(path):
    url = f"http://localhost:9090/api/v1/{path}"
    with urllib.request.urlopen(url, timeout=5) as r:
        return json.loads(r.read())

try:
    rules = api("rules")
    rec   = [r["name"] for g in rules["data"]["groups"]
             for r in g["rules"] if r["type"] == "recording"]
    alert = [r["name"] for g in rules["data"]["groups"]
             for r in g["rules"] if r["type"] == "alerting"]
    print(f"Recording rules ({len(rec)}):", ", ".join(rec) if rec else "（尚未載入）")
    print(f"Alert rules     ({len(alert)}):", ", ".join(alert) if alert else "（尚未載入）")
    if not rec:
        print("\n→ 確認 infra/prometheus/prometheus.yml 包含 rule_files 指向 infra/prometheus/alerts.yml")
        print("→ 或執行 Lab 08 完成部署步驟")
except Exception as e:
    print(f"Prometheus 未啟動或未回應：{e}")
